In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import pyarrow as pa

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *

# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# concat/fill fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap.
install_memory_guard()

# This notebook is the AGGREGATOR. Each upstream notebook (1..10) writes one
# Feature_data/<source>.parquet; here we concatenate every one of them into a
# single wide training matrix, aligned on the dispatch-price 5-min index (the
# fullest timeline). The combined file is what the downstream model pipeline
# (variables.FEATURES_DATASET_PATH) actually consumes, so re-run this whenever
# any upstream feature notebook is regenerated.
FEATURE_DIR = "Feature_data"
SPINE_NAME  = "1_dispatch_price.parquet"     # fullest 5-min index -> the spine
OUTPUT_NAME = os.path.basename(str(variables.FEATURES_DATASET_PATH))


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 4.9G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


In [2]:
import gc
import re
import resource
import pyarrow.parquet as pq

# Gather every feature table in Feature_data/ except the combined output itself.
paths = sorted(glob.glob(os.path.join(FEATURE_DIR, "*.parquet")))
paths = [p for p in paths if os.path.basename(p) != OUTPUT_NAME]

# The dispatch-price table spans the fullest 5-min timeline -> use it as spine.
spine_index = pd.read_parquet(os.path.join(FEATURE_DIR, SPINE_NAME), columns=[]).index
if spine_index.has_duplicates:
    dup = int(spine_index.duplicated(keep="last").sum())
    print(f"[spine] dropping {dup} duplicate timestamps (keeping last)")
    spine_index = spine_index[~spine_index.duplicated(keep="last")]

# --- Retain the informative core columns, drop the redundant ones ----------
# The predispatch/PDPASA sources store a full <signal>_<region>_h1..h78 forecast
# curve replicated across every region (+ TAS) and every interconnector. We only
# forecast one region, and each notebook already emits compact SUMMARY features
# for the neighbours, so the neighbour/TAS raw per-horizon curves are pure
# duplicates of information already present. We keep the target region's raw
# curves (the strongest per-horizon predictors) and the interconnector links
# that physically touch it; every other column (summaries, actuals, calendar,
# spreads) is kept as-is. This cuts ~3.8k redundant columns without losing signal.
REGION = variables.TARGET_REGION
REGION_INTERCONNECTORS = {
    "nsw": ["nsw1_qld1", "vic1_nsw1", "n_q_mnsp1"],
    "qld": ["nsw1_qld1", "n_q_mnsp1"],
    "vic": ["vic1_nsw1", "v_sa", "v_s_mnsp1", "t_v_mnsp1"],
    "sa":  ["v_sa", "v_s_mnsp1"],
}.get(REGION, [])
_H = re.compile(r"_h\d+$")


def _keep_column(col: str) -> bool:
    """Keep everything except non-target raw forecast-curve columns (…_h{k})."""
    m = _H.search(col)
    if not m:
        return True  # summaries, actuals, calendar, spreads -> always keep
    base = col[: m.start()]
    return base.endswith(f"_{REGION}") or any(base.endswith(ic) for ic in REGION_INTERCONNECTORS)


# Pass 1 (schemas only, no data): decide the kept column layout.
plan = []          # (path, [kept cols in this source])
col_names = []     # final column order
seen: set = set()
for p in paths:
    schema = pq.ParquetFile(p).schema_arrow
    md = schema.pandas_metadata or {}
    index_cols = {c for c in md.get("index_columns", []) if isinstance(c, str)}
    names = [n for n in schema.names if n not in index_cols]  # data cols only
    keep = [c for c in names if _keep_column(c) and c not in seen]
    seen.update(keep)
    plan.append((p, keep))
    col_names.extend(keep)

n_rows = len(spine_index)
n_cols = len(col_names)
print(f"Combined matrix: {n_rows:,} x {n_cols:,} = {n_rows * n_cols * 4 / 1e9:.1f} GB")

# The matrix is ~10 GB; holding it in RAM is what kills the kernel. Build it on a
# DISK-BACKED memmap instead. The memmap needs ~10 GB of VIRTUAL address space,
# which collides with the guard's RAM-sized cap and would turn an allocation into
# a hard abort -> lift the soft address-space limit to the hard limit for this
# build. Resident RAM stays bounded because (a) the memmap is COLUMN-MAJOR so each
# column-chunk write is contiguous (only that chunk's pages are dirtied, not the
# whole file), and (b) we flush after every source so those pages are written back
# and reclaimable.
_as_soft, _as_hard = resource.getrlimit(resource.RLIMIT_AS)
resource.setrlimit(resource.RLIMIT_AS, (_as_hard, _as_hard))

_MMAP_PATH = os.path.join(FEATURE_DIR, "_0_all_build.tmp.dat")
combined_mm = np.memmap(_MMAP_PATH, dtype=np.float32, mode="w+", shape=(n_rows, n_cols), order="F")

first_valid_pos = np.zeros(n_cols, dtype=np.int64)   # first non-NaN row per column
has_valid       = np.zeros(n_cols, dtype=bool)
nan_frac_inwin  = np.ones(n_cols, dtype=np.float32)  # NaN fraction in modelling window
in_scope = np.asarray(spine_index >= variables.PIPELINE_START_DATE)

COLCHUNK = 256
offset = 0
for p, keep in plan:
    if not keep:
        print(f"{os.path.basename(p):40s} kept     0")
        continue
    for cstart in range(0, len(keep), COLCHUNK):
        sub = keep[cstart:cstart + COLCHUNK]
        part = read_parquet_float32(p, columns=sub, verbose=False)
        if part.index.has_duplicates:
            part = part[~part.index.duplicated(keep="last")]
        part = part.reindex(spine_index)

        notna = part.notna().to_numpy()
        sl = slice(offset, offset + len(sub))
        has_valid[sl]       = notna.any(axis=0)
        first_valid_pos[sl] = notna.argmax(axis=0)
        nan_frac_inwin[sl]  = 1.0 - notna[in_scope].mean(axis=0)
        del notna

        combined_mm[:, sl] = part.ffill().to_numpy(dtype=np.float32)
        offset += len(sub)
        del part
        gc.collect()
    combined_mm.flush()   # write this source's dirty pages so RSS stays bounded
    print(f"{os.path.basename(p):40s} kept {len(keep):5d}")

combined_mm.flush()
print("Matrix built on disk:", _MMAP_PATH)


Combined matrix: 893,664 x 2,821 = 10.1 GB
10_cross_source.parquet                  kept    16
1_dispatch_price.parquet                 kept   645
2_dispatch_region_sum.parquet            kept   117
3_generation_fuel.parquet                kept    91
4_STTM_DWGM.parquet                      kept    21
5_weather.parquet                        kept   104
6_1_predispatch_price.parquet            kept   175
6_2_predispatch_region_sum.parquet       kept   484
6_3_predispatch_interconnector.parquet   kept   488
7_pdpasa_region_solution.parquet         kept   634
8_1_bid_stack.parquet                    kept    29
8_2_bid_prices.parquet                   kept    17
Matrix built on disk: Feature_data/_0_all_build.tmp.dat


Handle missing values

Downstream consumers (e.g. `mutual_info_regression`) can't handle NaNs. We only ever
forward-fill (never backward-fill) so no feature ever leaks a future value into the past.
Columns that are mostly missing, or whose data only starts partway through the modelling
window (`PIPELINE_START_DATE` onward), can't be forward-filled without a large unmodellable
gap, so those are dropped outright instead.

In [3]:
import pyarrow as pa

# Same drop rules as before, decided from the per-column stats gathered while
# building the matrix (no full-matrix boolean copy): drop a column that is >50%
# NaN inside the modelling window, starts >30 days after the window opens, or is
# never valid.
start_plus_30 = variables.PIPELINE_START_DATE + pd.Timedelta(days=30)
first_valid_ts = spine_index.to_numpy()[first_valid_pos]
too_late  = (~has_valid) | (first_valid_ts > np.datetime64(start_plus_30))
drop_mask = (nan_frac_inwin > 0.5) | too_late
keep_pos  = np.where(~drop_mask)[0]
kept_names = [col_names[i] for i in keep_pos]
print(f"Dropped {int(drop_mask.sum())} column(s) with insufficient in-window coverage")

# A row survives dropna() only once every kept column has started (forward-fill
# can't fill leading NaNs), so the cutoff is the latest first-valid row among the
# kept columns. Everything before it is dropped.
cutoff = int(first_valid_pos[keep_pos].max()) if len(keep_pos) else 0
print(f"Dropped {cutoff} leading row(s) that predate full feature history")

# Stream the kept sub-matrix (rows >= cutoff, kept columns) from the memmap
# straight to parquet in row batches; the full frame is never assembled in RAM.
out_path = os.path.join(FEATURE_DIR, OUTPUT_NAME)
writer = None
BATCH = 50_000
try:
    for r0 in range(cutoff, n_rows, BATCH):
        r1 = min(r0 + BATCH, n_rows)
        block = np.array(combined_mm[r0:r1][:, keep_pos], dtype=np.float32)
        bdf = pd.DataFrame(block, index=spine_index[r0:r1], columns=kept_names)
        table = pa.Table.from_pandas(bdf, preserve_index=True)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        del block, bdf, table
        gc.collect()
finally:
    if writer is not None:
        writer.close()
print(f"Saved {out_path} | rows {n_rows - cutoff:,} cols {len(kept_names):,}")


Dropped 6 column(s) with insufficient in-window coverage
Dropped 105192 leading row(s) that predate full feature history
Saved Feature_data/0_all_features.parquet | rows 788,472 cols 2,815


In [4]:
# Drop the on-disk build scratch file now the parquet is written.
try:
    del combined_mm
except NameError:
    pass
gc.collect()
if os.path.exists(_MMAP_PATH):
    os.remove(_MMAP_PATH)
    print("Removed scratch file", _MMAP_PATH)


Removed scratch file Feature_data/_0_all_build.tmp.dat


In [5]:
# Output is NaN-free by construction (forward-filled, pre-history rows dropped).
# Report yearly row coverage from the written file without loading it whole.
final_index = pd.read_parquet(os.path.join(FEATURE_DIR, OUTPUT_NAME), columns=[]).index
print(final_index.to_series().groupby(final_index.year).size().to_string())


Date
2019    105047
2020    105408
2021    105120
2022    105120
2023    105120
2024    105408
2025    105120
2026     52129


In [ ]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()
